In [1]:
!pip install datasets==2.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from datasets import load_dataset

ds = load_dataset("Hello-SimpleAI/HC3", "all")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Dataset new_dataset downloaded and prepared to /root/.cache/huggingface/datasets/Hello-SimpleAI___new_dataset/all/1.1.0/5af5910f9f3fe7aace30e32ad4c1ab776ca08183d00e9b2a091308549f69f683. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
import pandas as pd
df = ds['train'].to_pandas()

In [4]:
df

,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5
4,4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5
...,...,...,...,...,...
24317,24317,Is rise in pressure from 116/66 to 140/80 norm...,[Hello!Welcome and thank you for asking on HCM...,[It's not uncommon for blood pressure to fluct...,medicine
24318,24318,What could cause a painless lump in the right ...,"[Hi, * As per my surgical experience, the issu...",[There are several possible causes of a painle...,medicine
24319,24319,Can Acutret be given to a child for treatment ...,[Although it is difficult to comment whether A...,[It is not appropriate for me to recommend a s...,medicine
24320,24320,Are BP of 119/65 and pulse of 35 causes for co...,[Welcome and thank you for asking on HCM! I ha...,[It is not uncommon for people with rheumatoid...,medicine


In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source'],
        num_rows: 24322
    })
})

In [6]:
df.source.drop_duplicates()

,source
0,reddit_eli5
17112,open_qa
18299,wiki_csai
19141,finance
23074,medicine


In [7]:
def separate_dataset(df_train):#separate dataset at the fraction of 1 : 1 : 8 in each source into test, validation, and train data.
  #computing the fraction on each source to make equally distributed data.
  # making test dataset
  source_list = ['open_qa', 'wiki_csai', 'finance', 'medicine']
  df_source = df[df.source == 'reddit_eli5']
  df_test = df_source.sample(frac = 0.1, random_state= 1)
  drop_ids = list(df_test['id'].astype("int64").values)
  for source in source_list:
    df_source = df[df.source == source]
    df_source_test = df_source.sample(frac = 0.1, random_state= 1)
    df_test = pd.concat([df_test,df_source_test])
    drop_ids.extend(list(df_source_test['id'].astype("int64").values))

 #remove the test data id from full data
 # the data contains train and validation data.
  df_train = df.drop(drop_ids)

  # making val dataset
  df_source = df_train[df_train.source == 'reddit_eli5']
  df_val = df_source.sample(frac = 1/9, random_state= 1)
  drop_ids = list(df_val['id'].astype("int64").values)
  for source in source_list:
    df_source = df_train[df_train.source == source]
    df_source_val = df_source.sample(frac = 1/9, random_state= 1)
    df_val = pd.concat([df_val,df_source_val])
    drop_ids.extend(list(df_source_val['id'].astype("int64").values))

  #remove the validataion data id from train and validation data.
  df_train = df_train.drop(drop_ids)

  return df_test, df_val, df_train


In [8]:
# test data : validation data : training data = 8 : 1 : 1
df_test, df_val, df_train = separate_dataset(df) # separate with the source

In [9]:
df_train.count().id+ df_val.count() + df_test.count().id

,0
id,24322
question,24322
human_answers,24322
chatgpt_answers,24322
source,24322


In [10]:
def to_bert_input(df):#id, question, sentenece, label(human=0,gpt=1)
  df_human = df[['id', 'question', 'human_answers']]
  df_human = df_human.rename(columns = {"human_answers":"text"})
  df_human = df_human.assign(label = 0)
  df_gpt = df[['id', 'question', 'chatgpt_answers']]
  df_gpt = df_gpt.rename(columns = {"chatgpt_answers":"text"})
  df_gpt = df_gpt.assign(label = 1)
  return pd.concat([df_human[['text','label']],df_gpt[['text','label']]])

In [11]:
# into text: , label:
test_data = to_bert_input(df_test)
val_data = to_bert_input(df_val)
train_data = to_bert_input(df_train)


In [12]:
test_data

,text,label
1300,[In true 5 year old explanation form : Your pr...,0
6069,"[Well , are we talking actual lives or quality...",0
44,[Florida was chosen for several major reasons ...,0
16977,[Because it 's not actually a scandal . I hate...,0
15997,[Pretend you have a room full of chalk boards ...,0
...,...,...
24060,[It's difficult to say exactly what the dark o...,1
23237,[!\nnetwork error\n\n\n\nThere was an error ge...,1
24131,[Celiac disease is an autoimmune disorder that...,1
23344,[I'm sorry to hear that you have been experien...,1


In [13]:
#store the data into csv file
test_data.to_csv('test.csv', index = False)
val_data.to_csv('val.csv', index = False)
train_data.to_csv('train.csv', index = False)
